In [1]:
%load_ext autoreload
# autoreload 2：改了 benchmark/ 下的文件不用重启 kernel
%autoreload 2

import os
os.chdir(os.path.expanduser("~/projects/LLM/hw2"))   # benchmark 包以 hw2/ 为根

from benchmark import BenchConfig, run_config, nsys_profile, nsys_stats, snapshot

PROF = "profiles"                       # nsys 报告、显存快照（gitignored）
def assets(sec):                        # 每节的落盘目录 notes/assets/s<sec>；各节开头 OUT = assets(n)
    d = f"notes/assets/s{sec}"; os.makedirs(d, exist_ok=True); return d


## 0 Background Setups


只用 `MODEL_SIZES` 和几条公式，实测数（full step 时长、fwd_bwd 峰值）从 §2.1 的 `stages_b4_seq512.json` 读。结果存 `paper_estimates.txt`，写进 `blog.md` §0。
- 每层参数 = 4d² + 3·d·d_ff + 2d（q/k/v/o、SwiGLU 三矩阵、两个 RMSNorm），+ 2·V·d（embedding 与 lm_head 不共享）
- 前向 FLOPs/token = 2·(N − V·d) + 4·L·S·d；训练 = 3× 前向
- 训练静态显存 = 16 B/参数（fp32 权重 4 + 梯度 4 + Adam m/v 8）；激活（实测）= fwd_bwd 峰值 − 权重 − 梯度
- FLOPs/s = 训练 FLOPs/step ÷ 实测 full step；训练时长 = 20N token ÷ (token/step) × 步长

In [ ]:
OUT = assets(0)

In [2]:
import json
from benchmark import MODEL_SIZES

V, S, B = 10000, 512, 4
GiB = 2**30
meas = {(r["size"], r["mode"]): r for r in json.load(open(f"{assets(2)}/stages_b4_seq512.json")) if not r["inference"]}

rows = []
for size, c in MODEL_SIZES.items():
    d, dff, L = c["d_model"], c["d_ff"], c["num_layers"]
    N = L * (4*d*d + 3*d*dff + 2*d) + 2*V*d
    fwd_tok = 2*(N - V*d) + 4*L*S*d
    train_tok = 3 * fwd_tok
    w_gib, static_gib = N*4/GiB, N*16/GiB
    full, fb = meas.get((size, "full")), meas.get((size, "fwd_bwd"))
    ok = full and full["status"] == "OK"
    act = fb["peak_mem_gib"] - 2*w_gib if fb and fb["status"] == "OK" else None
    step_ms = full["avg_ms"] if ok else None
    flops_s = train_tok*B*S / (step_ms/1e3) if ok else None
    hours = 20*N / (B*S) * step_ms/1e3 / 3600 if ok else None
    rows.append(dict(size=size, N=f"{N/1e9:.2f}B", fwd_tok=f"{fwd_tok:.2e}", train_tok=f"{train_tok:.2e}",
                     fwd_step=f"{fwd_tok*B*S:.2e}", train_step=f"{train_tok*B*S:.2e}",
                     params_GiB=round(w_gib, 1), static_GiB=round(static_gib, 1),
                     act_GiB=round(act, 1) if act else "—", step_ms=step_ms or "OOM",
                     FLOPs_s=f"{flops_s:.1e}" if flops_s else "—", tok_20N=f"{20*N/1e9:.1f}B",
                     hours=f"{hours:.1f} h" if hours else "—"))
import pandas as pd
df = pd.DataFrame(rows).set_index("size"); display(df)
with open(f"{OUT}/paper_estimates.txt", "w") as f:
    f.write(f"vocab={V} seq={S} batch={B}  ({B*S} token/step)\n\n单位：FLOPs；吞吐为 FLOPs/s；显存 GiB (2^30 B)\n\n")
    f.write(df.to_string())

,N,fwd_tok,train_tok,fwd_step,train_step,params_GiB,static_GiB,act_GiB,step_ms,FLOPs_s,tok_20N,hours
size,,,,,,,,,,,,
small,0.13B,2.61e+08,7.82e+08,5.34e+11,1.60e+12,0.5,1.9,3.1,57.48,2.8e+13,2.6B,20.1 h
medium,0.42B,8.76e+08,2.63e+09,1.79e+12,5.38e+12,1.6,6.3,7.4,168.39,3.2e+13,8.5B,193.3 h
large,0.97B,2.01e+09,6.02e+09,4.11e+12,1.23e+13,3.6,14.4,13.1,384.12,3.2e+13,19.4B,1010.1 h
xl,3.41B,6.93e+09,2.08e+10,1.42e+13,4.26e+13,12.7,50.8,—,OOM,—,68.1B,—
10B,12.83B,2.60e+10,7.81e+10,5.33e+13,1.60e+14,47.8,191.2,—,OOM,—,256.7B,—


## 2 Profiling and Benchmarking

所有实验从这个 notebook 直接跑，入口都在 `benchmark` 包里：`sweep`（进程内批量计时）、`nsys_profile`（nsys 包子进程）、`snapshot`（显存快照子进程）。
产出：表 `.md` + 原始耗时 `.json` 落到 `notes/assets/s2/`，nsys 报告和快照落到 `profiles/`。写作在 `notes/blog.md`。
总耗时（5090）：2.1 ≈ 5 min，2.2 ≈ 10 min，2.4 ≈ 2 min，2.5 ≈ 3 min。

In [3]:
OUT = assets(2)

### 2.1 Benchmarking script

**(c) 不预热会怎样**：small/medium/large × warmup {0,1,2,5}，首步比稳态慢多少、对均值/标准差污染多大；warmup=1 之后还有没有残余。
`isolate=True` 必须开：不预热的开销大半是进程级一次性成本（kernel 懒加载、cuBLAS 句柄、显存池首次 cudaMalloc），同进程连跑会白捡前面的预热。

⚠️ **重启 kernel 后第一个跑这格**。前面的 setup / §0 都不碰 cuda，此时 kernel 没有 CUDA context，子进程独享整张卡；一旦 kernel 碰过 cuda（比如先跑了 (b)），它常驻的 ~0.7 GiB context 加上子进程自己的 context 会让 large full（峰值 27.5 GiB）差 0.5 GiB 假 OOM。

In [4]:
run_config("warmup_by_size", f"{OUT}/warmup_by_size_full.md", isolate=True)

已写出：notes/assets/s2/warmup_by_size_full.md                  
         notes/assets/s2/warmup_by_size_full.json（含逐步原始耗时）


,model,size,seq_len,batch,warmup,steps,mode,inference,autocast,avg_ms,std_ms,first_ms,rest_avg_ms,peak_mem_gib,status
0,basics,small,512,4,0,10,full,False,False,89.98,104.81,388.28,56.84,5.04,OK
1,basics,small,512,4,1,10,full,False,False,56.81,0.41,56.88,56.80,5.04,OK
2,basics,small,512,4,2,10,full,False,False,57.08,0.32,56.83,57.10,5.04,OK
3,basics,small,512,4,5,10,full,False,False,57.10,0.48,57.76,57.03,5.04,OK
4,basics,medium,512,4,0,10,full,False,False,200.76,99.50,483.91,169.29,13.74,OK
5,basics,medium,512,4,1,10,full,False,False,169.91,1.33,169.26,169.98,13.74,OK
6,basics,medium,512,4,2,10,full,False,False,171.46,2.10,172.26,171.37,13.74,OK
7,basics,medium,512,4,5,10,full,False,False,170.80,1.56,168.76,171.03,13.74,OK
8,basics,large,512,4,0,10,full,False,False,412.29,86.84,659.42,384.83,27.51,OK
9,basics,large,512,4,1,10,full,False,False,382.00,2.19,378.66,382.37,27.51,OK


**(b) 各阶段耗时**：5 size × {forward/no_grad, forward, fwd_bwd, full} 的墙钟均值 ± 标准差；backward / optimizer 靠相减。进程内跑，kernel 从这格起持有 CUDA context。
xl 在 forward（保留计算图）就 OOM、10B 建模型即 OOM——是 §3 checkpointing 和 §4 FlashAttention 的伏笔，OOM 本身就是数据。

In [4]:
run_config("default", f"{OUT}/stages_b4_seq512.md")

已写出：notes/assets/s2/stages_b4_seq512.md                     
         notes/assets/s2/stages_b4_seq512.json（含逐步原始耗时）


,model,size,seq_len,batch,warmup,steps,mode,inference,autocast,avg_ms,std_ms,first_ms,rest_avg_ms,peak_mem_gib,status
0,basics,small,512,4,5,10,forward,True,False,17.65,0.12,17.42,17.67,0.72,OK
1,basics,small,512,4,5,10,forward,False,False,17.88,0.15,17.85,17.89,3.98,OK
2,basics,small,512,4,5,10,fwd_bwd,False,False,53.72,0.21,53.40,53.75,4.08,OK
3,basics,small,512,4,5,10,full,False,False,57.43,0.27,57.24,57.45,5.04,OK
4,basics,medium,512,4,5,10,forward,True,False,50.95,0.22,50.83,50.97,1.90,OK
5,basics,medium,512,4,5,10,forward,False,False,51.37,0.28,51.16,51.39,10.49,OK
6,basics,medium,512,4,5,10,fwd_bwd,False,False,160.87,0.62,161.31,160.82,10.58,OK
7,basics,medium,512,4,5,10,full,False,False,172.02,1.25,172.49,171.97,13.74,OK
8,basics,large,512,4,5,10,forward,True,False,121.08,0.43,121.23,121.07,4.11,OK
9,basics,large,512,4,5,10,forward,False,False,120.27,0.42,120.74,120.22,20.19,OK


### 2.2 Nsight Systems

nsys 只能从进程启动时开始采集，所以 `nsys_profile` 内部起子进程；每份报告旁留 `.log`，里面有 benchmark 自己那行 timeit 结果——(a) 拿它和 nsys 的数对账。
`--trace=cuda,nvtx` 是轻量档；要 aten 算子级细节另加 `--pytorch=functions-trace,autograd-shapes-nvtx`，但观测开销明显变大。

**(a)–(d)** small/medium × seq {256,512,1024}，每档采一份 full：
(a) NVTX forward range 宽度 vs timeit（profiler 系统性偏高 2–8%）；(b) `--filter-nvtx=forward` 下 GPU 时间最长的 kernel，加上 backward 后是否还是它；(c) 非 matmul kernel 占前向多少、随 seq 怎么变；(d) forward range vs step range 的 matmul 占比。
选型：PDF 要"两个 size × 三个 >128 的 2 的幂，最大档取显存装得下的最长"。large 只到 512 凑不齐三档，所以 small+medium，1024 是两者上限。
一份 full 就够：实测 medium@1024 no_grad 前向与训练步中的前向 kernel 次数完全相同（1244），no_grad 省的是 CPU 侧建图。

In [5]:
for size in ["small", "medium"]:
    for seq in [256, 512, 1024]:
        nsys_profile(BenchConfig(size, seq, "full", nvtx=True, warmup=5, steps=5), f"{PROF}/{size}_seq{seq}_full")

✓ small   seq=256   full     → profiles/small_seq256_full.nsys-rep
✓ small   seq=512   full     → profiles/small_seq512_full.nsys-rep
✓ small   seq=1024  full     → profiles/small_seq1024_full.nsys-rep
✓ medium  seq=256   full     → profiles/medium_seq256_full.nsys-rep
✓ medium  seq=512   full     → profiles/medium_seq512_full.nsys-rep
✗ (见 .log) medium  seq=1024  full     → profiles/medium_seq1024_full.nsys-rep


**(e) attention 内 scores / softmax / matmul 三段占比**：`nvtx_attn=True` 把 SDPA 换成带三段 phase 的等价实现，每段结尾 sync，range 宽度 = GPU 耗时。
只看段间比，不拿墙钟对 §2.1（串行化后 forward 慢约 7%）。反向不会触发这些 range（autograd 在工作线程跑 grad_fn）。

In [6]:
for size in ["small", "medium"]:
    for seq in [256, 512, 1024]:
        nsys_profile(BenchConfig(size, seq, "forward", nvtx=True, nvtx_attn=True, warmup=5, steps=5), f"{PROF}/{size}_seq{seq}_attn")

✓ small   seq=256   forward  → profiles/small_seq256_attn.nsys-rep
✓ small   seq=512   forward  → profiles/small_seq512_attn.nsys-rep
✓ small   seq=1024  forward  → profiles/small_seq1024_attn.nsys-rep
✓ medium  seq=256   forward  → profiles/medium_seq256_attn.nsys-rep
✓ medium  seq=512   forward  → profiles/medium_seq512_attn.nsys-rep
✓ medium  seq=1024  forward  → profiles/medium_seq1024_attn.nsys-rep


取数：`nsys_stats(rep, report, filter_nvtx)` 返回 DataFrame。(a) 看 `nvtx_sum` 的 forward 行对 timeit；(b)(c)(d) 看 `cuda_gpu_kern_sum`，`filter_nvtx="forward"` 只统计前向内的 kernel，不加则是整步。

In [7]:
from benchmark import nsys_stats

rep = f"{PROF}/medium_seq512_full"
display(nsys_stats(rep, "nvtx_sum"))                                        # (a) forward range 均值 vs timeit
display(nsys_stats(rep, "cuda_gpu_kern_sum", filter_nvtx="forward").head(8))  # (b)(c) 前向内 top kernel
display(nsys_stats(rep, "cuda_gpu_kern_sum").head(8))                         # (d) 整步 top kernel
display(nsys_stats(f"{PROF}/medium_seq512_attn", "nvtx_sum"))                 # (e) attn.scores / softmax / matmul

,Time (%),Instances,Style,Range,Total Time (ms),Avg (ms),Med (ms),Min (ms),Max (ms),StdDev (ms)
0,41.3,1,PushPop,:warmup,1232.556052,1232.556052,1232.556052,1232.556052,1232.556052,0.000000
1,29.4,5,PushPop,:step,877.201645,175.440329,174.765296,173.954785,177.461457,1.445390
2,18.6,5,PushPop,:backward,555.472211,111.094442,110.673763,110.269359,112.924551,1.065144
3,8.6,5,PushPop,:forward,256.256396,51.251279,51.241982,50.861864,51.623452,0.359971
4,1.9,5,PushPop,:optimizer,57.481094,11.496219,11.469457,10.995347,11.939914,0.350448
5,0.1,5,PushPop,:zero_grad,3.667007,0.733401,0.462695,0.372618,1.877262,0.641020
6,0.0,10,PushPop,CCCL:cub::DeviceRadixSort,0.576658,0.057666,0.059548,0.040509,0.062743,0.006446


,Time (%),Instances,Name,Total Time (ms),Avg (ms),Med (ms),Min (ms),Max (ms),StdDev (ms)
0,49.1,73,void cutlass::Kernel2<cutlass_80_simt_sgemm_12...,24.151049,0.330836,0.302176,0.279296,0.787585,0.105316
1,18.4,96,void cutlass::Kernel2<cutlass_80_simt_sgemm_25...,9.038763,0.094154,0.091488,0.090560,0.333792,0.024755
2,4.6,24,"void magma_sgemmEx_kernel<float, float, float,...",2.255684,0.093987,0.072160,0.071648,0.595169,0.106752
3,3.2,24,"void at::native::elementwise_kernel<(int)128, ...",1.567748,0.065323,0.052464,0.048096,0.352641,0.061287
4,3.0,24,void cutlass::Kernel2<cutlass_80_simt_sgemm_64...,1.457092,0.060712,0.060736,0.059264,0.062816,0.000847
5,2.9,25,void at::native::vectorized_elementwise_kernel...,1.405952,0.056238,0.054400,0.047808,0.100224,0.009971
6,2.7,48,void at::native::vectorized_elementwise_kernel...,1.343330,0.027986,0.024976,0.015360,0.354177,0.048317
7,2.7,25,"void at::native::elementwise_kernel<(int)128, ...",1.327970,0.053119,0.052448,0.046752,0.076769,0.005616


,Time (%),Instances,Name,Total Time (ms),Avg (ms),Med (ms),Min (ms),Max (ms),StdDev (ms)
0,16.9,1690,void cutlass::Kernel2<cutlass_80_simt_sgemm_25...,281.009999,0.166278,0.080416,0.078176,0.911138,0.116982
1,14.3,730,void cutlass::Kernel2<cutlass_80_simt_sgemm_12...,237.683848,0.325594,0.301728,0.277377,0.979681,0.099789
2,12.1,730,void cutlass::Kernel2<cutlass_80_simt_sgemm_12...,201.750856,0.276371,0.252769,0.246177,0.888065,0.088366
3,7.8,3880,void at::native::vectorized_elementwise_kernel...,130.053418,0.033519,0.008480,0.000800,0.759169,0.053954
4,5.6,960,"void at::native::elementwise_kernel<(int)128, ...",93.056530,0.096934,0.088256,0.046464,0.786561,0.082389
5,5.6,960,void cutlass::Kernel2<cutlass_80_simt_sgemm_25...,92.866256,0.096736,0.091457,0.089953,0.570272,0.039561
6,5.0,8040,void at::native::vectorized_elementwise_kernel...,83.189902,0.010347,0.004864,0.000832,0.640289,0.025987
7,4.7,960,void cutlass::Kernel2<cutlass_80_simt_sgemm_12...,78.499869,0.081771,0.080128,0.075392,0.441601,0.017481


,Time (%),Instances,Style,Range,Total Time (ms),Avg (ms),Med (ms),Min (ms),Max (ms),StdDev (ms)
0,43.7,1,PushPop,:warmup,563.029087,563.029087,563.029087,563.029087,563.029087,0.000000
1,20.9,5,PushPop,:step,268.563413,53.712683,53.688645,53.012657,54.180152,0.452450
2,20.5,5,PushPop,:forward,264.425325,52.885065,52.940263,52.316322,53.357357,0.403838
3,11.1,120,PushPop,:attn.scores,143.442217,1.195352,1.211494,0.268093,1.562821,0.195557
4,2.5,120,PushPop,:attn.softmax,31.749114,0.264576,0.218837,0.201514,0.596132,0.087896
5,1.3,120,PushPop,:attn.matmul,16.717685,0.139314,0.111030,0.099410,0.370623,0.062071


### 2.3 Mixed precision accumulation

实验就在下面这个 cell，输出存 `accumulation_1000x0.01.txt`。结论：精度由**累加器** dtype 决定——fp16 累加漂到 9.95，bf16 累加卡死在 4.0，fp32 累加无论加数是什么都在 10.00x。

In [8]:
import torch, pandas as pd

def acc(s_dtype, x_dtype, cast=False):
    s = torch.tensor(0, dtype=s_dtype)
    x = torch.tensor(0.01, dtype=x_dtype)
    for _ in range(1000):
        s += x.type(s_dtype) if cast else x
    return s.item()

f32, f16, bf16 = torch.float32, torch.float16, torch.bfloat16
rows = [("s(fp32) += x(fp32)", acc(f32, f32), "")]
for name, dt in [("fp16", f16), ("bf16", bf16)]:
    rows += [(f"s({name}) += x({name})",           acc(dt, dt),            ""),
             (f"s(fp32) += x({name})",             acc(f32, dt),           "x 自动升到 fp32 再加"),
             (f"s(fp32) += x({name}).type(fp32)",  acc(f32, dt, cast=True), "x 手动升到 fp32 再加")]
df = pd.DataFrame(rows, columns=["累加", "结果", "说明"])
df["误差"] = ((df["结果"] - 10) / 10).map("{:+.2%}".format)
df["结果"] = df["结果"].map("{:.4f}".format)
stored = pd.DataFrame({"dtype": ["fp32", "fp16", "bf16"],
                       "0.01 存成": [f"{torch.tensor(0.01, dtype=d).item():.10f}" for d in (f32, f16, bf16)]})
display(df.set_index("累加")); display(stored.set_index("dtype"))
with open(f"{OUT}/accumulation_1000x0.01.txt", "w") as f:
    f.write("s = 0; 重复 1000 次 s += 0.01   正确答案 = 10.0\n\n" + df.to_string(index=False) + "\n\n" + stored.to_string(index=False) + "\n")

,结果,说明,误差
累加,,,
s(fp32) += x(fp32),10.0001,,+0.00%
s(fp16) += x(fp16),9.9531,,-0.47%
s(fp32) += x(fp16),10.0021,x 自动升到 fp32 再加,+0.02%
s(fp32) += x(fp16).type(fp32),10.0021,x 手动升到 fp32 再加,+0.02%
s(bf16) += x(bf16),4.0000,,-60.00%
s(fp32) += x(bf16),10.0098,x 自动升到 fp32 再加,+0.10%
s(fp32) += x(bf16).type(fp32),10.0098,x 手动升到 fp32 再加,+0.10%


,0.01 存成
dtype,
fp32,0.0099999998
fp16,0.0100021362
bf16,0.0100097656


### 2.4 Benchmarking mixed precision

**(a)(b)** ToyModel（PDF 原样）在 fp16 / bf16 autocast 下各组件的 dtype，输出存 `autocast_dtypes.txt`。`backward()` 放在 autocast 外。

In [9]:
import torch, pandas as pd
from torch import nn

class ToyModel(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, 10, bias=False)
        self.ln = nn.LayerNorm(10)
        self.fc2 = nn.Linear(10, out_features, bias=False)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.relu(self.fc1(x)); self._fc1 = x
        x = self.ln(x);             self._ln = x
        return self.fc2(x)

def probe(dtype):
    m = ToyModel(16, 4).cuda()
    x, y = torch.randn(8, 16, device="cuda"), torch.randint(0, 4, (8,), device="cuda")
    r = {"参数（autocast 外）": m.fc1.weight.dtype}
    with torch.autocast("cuda", dtype=dtype):
        r["参数（autocast 内）"] = m.fc1.weight.dtype
        logits = m(x)
        loss = nn.functional.cross_entropy(logits, y)
    loss.backward()   # backward 在 autocast 外：反向沿用前向记下的 dtype，最终 .grad 是 fp32
    r |= {"fc1 输出": m._fc1.dtype, "ln 输出": m._ln.dtype, "logits（fc2 输出）": logits.dtype,
          "loss": loss.dtype, "梯度（.grad）": m.fc1.weight.grad.dtype, "ln 的参数梯度": m.ln.weight.grad.dtype}
    return r

res = {"autocast=fp16": probe(torch.float16), "autocast=bf16": probe(torch.bfloat16)}
df = pd.DataFrame(res).map(lambda d: str(d).removeprefix("torch.")); df.index.name = "组件"
display(df)
open(f"{OUT}/autocast_dtypes.txt", "w").write(df.to_string() + "\n")

,autocast=fp16,autocast=bf16
组件,,
参数（autocast 外）,float32,float32
参数（autocast 内）,float32,float32
fc1 输出,float16,bfloat16
ln 输出,float32,float32
logits（fc2 输出）,float16,bfloat16
loss,float32,float32
梯度（.grad）,float32,float32
ln 的参数梯度,float32,float32


430

**(c) fp32 vs bf16 autocast**：small/medium/large/xl × {forward, fwd_bwd} 耗时与峰值显存，加速比随模型大小的趋势。
fp32 基准是 `allow_tf32=False`（SIMT GEMM），bf16 走 Tensor Core，加速比 = 位宽减半 + 硬件路径切换两个效应叠加。

In [10]:
run_config("mixed_precision", f"{OUT}/mixed_precision_b4_seq512.md")

已写出：notes/assets/s2/mixed_precision_b4_seq512.md            
         notes/assets/s2/mixed_precision_b4_seq512.json（含逐步原始耗时）


,model,size,seq_len,batch,warmup,steps,mode,inference,autocast,avg_ms,std_ms,first_ms,rest_avg_ms,peak_mem_gib,status
0,basics,small,512,4,5,10,forward,False,False,17.37,0.31,17.88,17.31,3.99,OK
1,basics,small,512,4,5,10,fwd_bwd,False,False,52.40,0.39,52.71,52.37,4.08,OK
2,basics,small,512,4,5,10,forward,False,True,9.04,0.36,8.84,9.06,3.16,OK
3,basics,small,512,4,5,10,fwd_bwd,False,True,29.18,0.42,29.39,29.15,3.18,OK
4,basics,medium,512,4,5,10,forward,False,False,50.54,0.46,50.40,50.55,10.49,OK
5,basics,medium,512,4,5,10,fwd_bwd,False,False,156.47,0.65,156.54,156.46,10.58,OK
6,basics,medium,512,4,5,10,forward,False,True,25.36,0.40,25.99,25.29,8.34,OK
7,basics,medium,512,4,5,10,fwd_bwd,False,True,84.91,0.75,84.67,84.94,8.36,OK
8,basics,large,512,4,5,10,forward,False,False,119.66,1.23,118.92,119.74,20.19,OK
9,basics,large,512,4,5,10,fwd_bwd,False,False,353.28,1.67,351.72,353.45,20.28,OK


**(d) 验证：bf16 峰值变化 = 权重副本(+) + activation 变化(−)**。用 `saved_tensors_hooks` 把 small/medium/large @512 fwd_bwd 为反向存的张量按来源数一遍，
把「存的是参数本身」「bf16 权重副本」「activation」分开；输出存 `autocast_saved_tensors.txt`，填进 blog (d) 表后两列。

In [ ]:
import torch
from collections import defaultdict
from benchmark.model_bench import build_model, build_batch
from cs336_basics.nn_utils import cross_entropy

GiB = 1024**3
def saved_by_kind(size, autocast):
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    cfg = BenchConfig(size, 512, "fwd_bwd")
    model = build_model(cfg); x, y = build_batch(cfg)
    seen = {}                                   # data_ptr -> (nbytes, dtype)，同一张量被多个 op 存只算一次
    def pack(t):
        seen.setdefault(t.data_ptr(), (t.numel() * t.element_size(), t.dtype)); return t
    with torch.autograd.graph.saved_tensors_hooks(pack, lambda t: t):
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=autocast):
            loss = cross_entropy(model(x), y)
    torch.cuda.synchronize()
    param_ptrs = {p.data_ptr() for p in model.parameters()}
    param_numels = {p.numel() for p in model.parameters()}
    params_saved = sum(nb for ptr, (nb, dt) in seen.items() if ptr in param_ptrs)          # fp32 W 本身，已在权重里
    copies = sum(nb for nb, dt in seen.values() if dt == torch.bfloat16 and nb // 2 in param_numels)  # autocast 造的 bf16 W
    act = sum(nb for nb, _ in seen.values()) - params_saved - copies
    by_dtype = defaultdict(int)
    for nb, dt in seen.values(): by_dtype[str(dt).replace('torch.', '')] += nb
    loss.backward(); torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated()
    line = (f"{size:6} autocast={autocast!s:5}: 存的参数本身 {params_saved/GiB:.2f} | bf16 权重副本 {copies/GiB:.2f} | "
            f"activation {act/GiB:.2f} ({ {k: round(v/GiB, 2) for k, v in by_dtype.items() if v > 1e6} }) | peak {peak/GiB:.2f} GiB")
    print(line)
    del model, x, y, loss; torch.cuda.empty_cache()
    return line, peak, copies, act

lines = []
for size in ["small", "medium", "large"]:
    l32, p32, _, a32 = saved_by_kind(size, False)
    l16, p16, c16, a16 = saved_by_kind(size, True)
    summary = f"{size:6} 峰值变化 {(p16-p32)/GiB:+.2f} GiB = 副本 {c16/GiB:+.2f} + activation 变化 {(a16-a32)/GiB:+.2f} (= {(c16+a16-a32)/GiB:+.2f})"
    print(summary); lines += [l32, l16, summary]
open(f"{OUT}/autocast_saved_tensors.txt", "w").write("\n".join(lines) + "\n")


### 2.5 Memory profiling

**(a)(e) 显存时间线**：xl × seq {128, 2048} × {forward/no_grad, full} → `.pickle`。
快照从预热开始记，`warmup=1, steps=1` 共两步：第一步带初始化噪声，看第二步的三个峰。xl@2048 full 预期第一个前向就 OOM，快照仍落盘。
看图：把 `profiles/mem_*.pickle` 拖进 https://pytorch.org/memory_viz（已画好的 png 在 `notes/assets/s2/mem_xl_*.png`）。

In [11]:
for seq in [128, 2048]:
    snapshot(BenchConfig("xl", seq, "forward", inference=True, warmup=1, steps=1), f"{PROF}/mem_xl_seq{seq}_forward.pickle")
    snapshot(BenchConfig("xl", seq, "full",                    warmup=1, steps=1), f"{PROF}/mem_xl_seq{seq}_full.pickle")
snapshot(BenchConfig("xl", 2048, "fwd_bwd", warmup=0, steps=1), f"{PROF}/mem_xl_seq2048_fwd_bwd.pickle")   # 预期 OOM，快照记录到炸掉为止


✓ xl      seq=128   forward  → profiles/mem_xl_seq128_forward.pickle
✓ xl      seq=128   full     → profiles/mem_xl_seq128_full.pickle
✓ xl      seq=2048  forward  → profiles/mem_xl_seq2048_forward.pickle
✓ xl      seq=2048  full     → profiles/mem_xl_seq2048_full.pickle


In [ ]:
# 从 pickle 画 active memory 曲线（横轴 = 分配事件序号；只记录了测量段的那 1 步）
from PIL import Image
from benchmark.memory import plot_snapshot
plot_snapshot(f"{PROF}/mem_xl_seq128_forward.pickle", f"{OUT}/mem_xl_seq128_forward.png",
    title="xl, seq 128 — ONE forward pass (no_grad), 32 layers", subtitle="warmup 1 / steps 1, only the measured step is recorded")
plot_snapshot(f"{PROF}/mem_xl_seq2048_forward.pickle", f"{OUT}/mem_xl_seq2048_forward.png",
    title="xl, seq 2048 — ONE forward pass (no_grad), 32 layers", subtitle="32 spikes = 32 layers; each spike = attention score-matrix chain, ~4 × 2 GiB live at once")
plot_snapshot(f"{PROF}/mem_xl_seq128_full.pickle", f"{OUT}/mem_xl_seq128_full.png", phases=True,   # 红线：按分配时的 Python 栈分 forward / backward / optimizer
    title="xl, seq 128 — ONE full step (forward + backward + AdamW), OOM at optimizer", subtitle="warmup 1 / steps 1; red lines = phase boundaries from the allocation's Python stack")
plot_snapshot(f"{PROF}/mem_xl_seq2048_fwd_bwd.pickle", f"{OUT}/mem_xl_seq2048_fwd_bwd.png", phases=True,
    title="xl, seq 2048 - fwd_bwd, OOM in layer 2's attention", subtitle="warmup 0 / steps 1, recorded until the OOM; layer 1 leaves ~4.7 GiB, layer 2's score-matrix chain hits 25.96")
ims = {n: Image.open(f"{OUT}/mem_xl_seq{n}.png") for n in ["128_forward", "2048_forward", "128_full", "2048_fwd_bwd"]}   # 2×2 拼一张
w = max(i.width for i in ims.values()); h = max(i.height for i in ims.values())
side = Image.new("RGB", (2 * w, 2 * h), "white")
side.paste(ims["128_forward"], (0, 0)); side.paste(ims["2048_forward"], (w, 0)); side.paste(ims["128_full"], (0, h)); side.paste(ims["2048_fwd_bwd"], (w, h))
side.save(f"{OUT}/mem_xl_timelines.png")


**峰值落在哪一刻（blog §2.2，图 2.2-1）**：用 hook 在每层前向结束 / 反向结束时读 `memory_allocated()`，得到与 M(j) 同横轴的实测点；再按公式画 W/G/A/T 堆叠曲线并把实测点叠上去。数字来自 `memory_xl_peak.md` 和 `autocast_saved_tensors.txt`。

In [ ]:
"""§2.2 图 2.2-1 的实测点：在每层 block 前向结束、反向结束时读 memory_allocated()，得到与 M(j) 同横轴的曲线。
small@512 / xl@128，fp32 / bf16 autocast，fwd_bwd 一步（预热一步后测第二步）。输出 peak_moment_measured.json。"""
import json, sys
import torch
from benchmark import BenchConfig
from benchmark.model_bench import build_model, build_batch
from cs336_basics.nn_utils import cross_entropy

GiB = 1024**3

def measure(size, seq, autocast):
    torch.cuda.empty_cache()
    cfg = BenchConfig(size, seq, "fwd_bwd")
    model = build_model(cfg); x, y = build_batch(cfg)
    layers = list(model.layers)
    fwd, bwd = [], []
    def rec(lst):
        torch.cuda.synchronize(); lst.append(torch.cuda.memory_allocated() / GiB)
    def fhook(m, inp, out):
        rec(fwd)
        # out 的梯度算出来的时刻 = 上一层（更靠近输出的那层）反向做完的时刻
        out.register_hook(lambda g: rec(bwd))
    for blk in layers:
        blk.register_forward_hook(fhook)
    def step():
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=autocast):
            loss = cross_entropy(model(x), y)
        loss.backward(); rec(bwd)                 # 最后一层（layer 0）反向做完
        model.zero_grad(set_to_none=True)
    step(); fwd.clear(); bwd.clear()            # 预热一步：cuBLAS 句柄等一次性分配不算进曲线
    torch.cuda.reset_peak_memory_stats()
    start = torch.cuda.memory_allocated() / GiB
    step()
    peak = torch.cuda.max_memory_allocated() / GiB
    del model, x, y; torch.cuda.empty_cache()
    return dict(size=size, seq=seq, autocast=autocast, start=start, fwd=fwd, bwd=bwd[1:], peak=peak)

out = []
for size, seq in [("small", 512), ("xl", 128)]:
    for ac in [False, True]:
        r = measure(size, seq, ac); out.append(r)
        print(f"{size}@{seq} autocast={ac}: start {r['start']:.2f} | fwd end {r['fwd'][-1]:.2f} | bwd end {r['bwd'][-1]:.2f} | peak {r['peak']:.2f}")
json.dump(out, open(f"{OUT}/peak_moment_measured.json", "w"), indent=1)


In [ ]:
"""§2.2：一步 fwd_bwd 的显存曲线，按 W / G / A / T 四项堆叠着色（fp32），bf16 autocast 画成线。
前向第 i 层结束：W + A·i/L（末尾加 T）；反向走完 j 层：W + G·j/L + A·(L−j)/L + T。
数字来自 memory_xl_peak.md / autocast_saved_tensors.txt。颜色与正文 🟦W 🟥G 🟩A 🟨T 一致。"""
import json
import matplotlib.pyplot as plt

MEAS = {(r["size"], r["autocast"]): r for r in json.load(open(f"{OUT}/peak_moment_measured.json"))}   # measure_peak_moment.py

C = {"W": "#4A90D9", "G": "#E4572E", "A": "#3CB371", "T": "#F2C14E"}
cases = [  # name, L, W, G, A_fp32, A_bf16(含副本), T, 实测峰值 fp32/bf16
    ("xl @ seq 128, batch 4  (G > A)",    32, 12.7, 12.7, 5.3,  3.5 + 6.35, 0.2,  25.56, 25.55),
    ("small @ seq 512, batch 4  (A > G)", 12, 0.48, 0.48, 3.41, 2.28 + 0.23, 0.17, 4.08, 3.18),
]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, (name, L, W, G, A, A16, T, p32, p16) in zip(axes, cases):
    xs = list(range(2 * L + 1))
    # 四项随 x 的值：前向 i=0..L，反向 j=1..L
    w = [W] * (2 * L + 1)
    a = [A * i / L for i in range(L + 1)] + [A * (L - j) / L for j in range(1, L + 1)]
    g = [0] * (L + 1) + [G * j / L for j in range(1, L + 1)]
    t = [0] * L + [T] * (L + 1)
    tot = [w[i] + a[i] + g[i] + t[i] for i in xs]
    ax.stackplot(xs, w, a, g, t, colors=[C["W"], C["A"], C["G"], C["T"]],
                 labels=["W weights", "A saved tensors", "G gradients (.grad)", "T backward temporaries"], alpha=.85)
    kp = max(xs, key=lambda x: tot[x])
    ax.annotate(f"fp32 peak {tot[kp]:.2f} (measured {p32})", (kp, tot[kp]), textcoords="offset points",
                xytext=(-8, 6) if kp > L else (8, 6), ha="right" if kp > L else "left", fontsize=8)
    # bf16 autocast：A 换成 A16（含副本），其余不变
    m16 = [W + A16 * i / L for i in range(L)] + [W + A16 + T] + [W + G * j / L + A16 * (L - j) / L + T for j in range(1, L + 1)]
    ax.plot(xs, m16, c="k", ls="--", lw=1.2, label="bf16 autocast")
    kp16 = max(xs, key=lambda x: m16[x])
    ax.annotate(f"bf16 peak {m16[kp16]:.2f} (measured {p16})", (kp16, m16[kp16]), textcoords="offset points",
                xytext=(-8, -14) if kp16 > L else (8, -14), ha="right" if kp16 > L else "left", fontsize=8)
    # 实测点：每层前向结束 / 反向结束时的 memory_allocated()（measure_peak_moment.py，hook 采样）
    size = name.split()[0]
    for ac, c, lab in [(False, "k", "measured fp32"), (True, "gray", "measured bf16")]:
        r = MEAS.get((size, ac))
        if r:
            pts = r["fwd"] + r["bwd"]
            ax.scatter(range(1, len(pts) + 1), pts, s=10, c=c, marker="x", zorder=5, label=lab)
    ax.axvline(L, c="gray", lw=.8, ls=":")
    ax.set_title(f"{name}\nW = G = {W} GiB, A = {A} GiB, L = {L}", fontsize=10)
    ax.set_xticks([0, L, 2 * L]); ax.set_xticklabels(["start", "forward ends\nbackward starts (j=0)", "backward ends (j=L)"], fontsize=8)
    ax.set_ylabel("active memory  GiB")
    ax.legend(fontsize=7.5, loc="upper left", framealpha=.9)
fig.suptitle("one fwd_bwd step, stacked by term: forward W + A·i/L, backward M(j) = W + G·j/L + A·(L−j)/L + T", fontsize=9.5)
fig.tight_layout()
fig.savefig(f"{OUT}/peak_moment.png", dpi=140)


**(f) nsys 显存 trace**：单层 TransformerBlock 为反向存了多少。
block{i} 前向 range 内分配、到 range 结束还没释放的字节 = 这层的 residual；按包着它们的 aten range 归组 → top 5 算子；反向该层净变化 Δ → 梯度 = Δ + residual（预期 ≈ 每层参数 × 4 B = 400 MiB）。
`memory=True` 会设 `PYTORCH_NO_CUDA_MEMORY_CACHING=1`：否则 caching allocator 从池里复用，nsys 只看到池增长。会慢很多，所以 warmup 0 / steps 1，xl@128 fwd_bwd（full 会 OOM）。

In [12]:
rep = f"{PROF}/mem_trace_xl_seq128_fwd_bwd"
nsys_profile(BenchConfig("xl", 128, "fwd_bwd", nvtx=True, nvtx_ops=True, warmup=0, steps=1), rep, memory=True)

✓ xl      seq=128   fwd_bwd  → profiles/mem_trace_xl_seq128_fwd_bwd.nsys-rep


True

In [13]:
# 归因 + 三联图（整步曲线 / block5 放大 / 对齐的 aten 算子行）
from benchmark.memory import block_residuals, plot_block
block_residuals(f"{rep}.nsys-rep", block=5)
plot_block(f"{rep}.nsys-rep", block=5, out=f"{OUT}/nsys_block5_memory.png")

block5  前向 4.5 ms，aten 算子 341 个，seq 826–990
  前向分配 288.3 MiB，其中活到 range 结束的（residual）166.3 MiB / 25 个张量
  top 5 来源：
        60.0 MiB   36.1%  aten::mul
        40.0 MiB   24.1%  aten::bmm
        20.0 MiB   12.0%  aten::empty
        20.0 MiB   12.0%  aten::sigmoid
        10.0 MiB    6.0%  aten::add
  反向窗口 11.5 ms：分配 1203.2，释放 954.4（其中 residual 161.3），净 +248.8 MiB
  ⇒ 反向新产生的张量（梯度等）≈ 净变化 + 释放的 residual = 410.0 MiB


/home/yc/projects/LLM/hw2/benchmark/memory.py:230: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


图已写出：notes/assets/s2/nsys_block5_memory.png


PosixPath('notes/assets/s2/nsys_block5_memory.png')

**(b)(c) 峰值显存**：xl × seq {128, 2048} × {forward, full} × {fp32, bf16} → `memory_xl_peak.md`。
`torch.cuda.max_memory_allocated()`（预热后清零，只统计测量段），单位 GiB。

In [14]:
run_config("memory_xl", f"{OUT}/memory_xl_peak.md")

已写出：notes/assets/s2/memory_xl_peak.md                       
         notes/assets/s2/memory_xl_peak.json（含逐步原始耗时）


,model,size,seq_len,batch,warmup,steps,mode,inference,autocast,avg_ms,std_ms,first_ms,rest_avg_ms,peak_mem_gib,status
0,basics,xl,128,4,2,2,forward,True,False,80.28,2.52,78.50,82.07,12.91,OK
1,basics,xl,128,4,2,2,full,False,False,NaN,NaN,NaN,NaN,29.25,OOM (optimizer)
2,basics,xl,128,4,2,2,forward,True,True,35.55,0.21,35.40,35.70,19.19,OK
3,basics,xl,128,4,2,2,full,False,True,NaN,NaN,NaN,NaN,29.24,OOM (optimizer)
4,basics,xl,2048,4,2,2,forward,True,False,2084.46,33.38,2060.85,2108.06,21.39,OK
5,basics,xl,2048,4,2,2,full,False,False,NaN,NaN,NaN,NaN,25.97,OOM (forward)
6,basics,xl,2048,4,2,2,forward,True,True,1074.68,3.80,1077.37,1072.00,25.28,OK
7,basics,xl,2048,4,2,2,full,False,True,NaN,NaN,NaN,NaN,26.79,OOM (forward)


---

# 3 Single-GPU Memory

In [2]:
OUT = assets(3)

## 3.1 Autograd Residuals

`saved_tensors_hooks` 看每个算子为反向 pack 了什么。逐算子 RMSNorm：6 次 Saving、4 块内存，额外的只有 `r`（8 KiB）和 `x̂`（20 MiB）。分析和图在 `blog.md` §3.1。

In [4]:
import torch
from torch import nn

In [16]:
class RMSNorm(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        eps: float = 1e-5,
        device=None,
    ):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size, device=device))
        self.eps = eps

    def forward(self, x):
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        x = x * rms
        return self.weight * x


In [17]:
def pack_hook(t):
    shape, dtype, grad_fn, ptr = t.shape, t.dtype, t.grad_fn, hex(t.data_ptr())
    print(f"Saving residual: {shape=}, {dtype=}, {grad_fn=}, {ptr=}")
    return t

def unpack_hook(t):
    shape, dtype, grad_fn, ptr = t.shape, t.dtype, t.grad_fn, hex(t.data_ptr())
    print(f"Loading residual: {shape=}, {dtype=}, {grad_fn=}, {ptr=}")
    return t

In [18]:
x = torch.randn((4, 512, 2560), requires_grad=True)

ln = RMSNorm(x.shape[-1])
with torch.autograd.graph.saved_tensors_hooks(pack_hook, unpack_hook):
    y = ln(x)
y.sum().backward()

Saving residual: shape=torch.Size([4, 512, 2560]), dtype=torch.float32, grad_fn=None, ptr='0x7325d0aa9040'
Saving residual: shape=torch.Size([4, 512, 1]), dtype=torch.float32, grad_fn=<RsqrtBackward0 object at 0x7325d1edf940>, ptr='0x2f2c6b00'
Saving residual: shape=torch.Size([4, 512, 1]), dtype=torch.float32, grad_fn=<RsqrtBackward0 object at 0x7325d1edf940>, ptr='0x2f2c6b00'
Saving residual: shape=torch.Size([4, 512, 2560]), dtype=torch.float32, grad_fn=None, ptr='0x7325d0aa9040'
Saving residual: shape=torch.Size([4, 512, 2560]), dtype=torch.float32, grad_fn=<MulBackward0 object at 0x7325d1edf940>, ptr='0x2f3b3c00'
Saving residual: shape=torch.Size([2560]), dtype=torch.float32, grad_fn=None, ptr='0x2ed01000'
Loading residual: shape=torch.Size([4, 512, 2560]), dtype=torch.float32, grad_fn=<MulBackward0 object at 0x732604116530>, ptr='0x2f3b3c00'
Loading residual: shape=torch.Size([2560]), dtype=torch.float32, grad_fn=None, ptr='0x2ed01000'
Loading residual: shape=torch.Size([4, 512, 

`torch.compile` 把 ①–⑤ 变成一个 autograd 节点：Saving 只剩 `x`、`w`、`r` 三条，`x̂` 反向现场重算。

In [19]:
x = torch.randn((4, 512, 2560), requires_grad=True)

# fused kernel
ln = torch.compile(RMSNorm(x.shape[-1]))
with torch.autograd.graph.saved_tensors_hooks(pack_hook, unpack_hook):
    y = ln(x)
    y.sum().backward()

Saving residual: shape=torch.Size([4, 512, 2560]), dtype=torch.float32, grad_fn=None, ptr='0x31bb3c80'
Saving residual: shape=torch.Size([2560]), dtype=torch.float32, grad_fn=None, ptr='0x2ec6b3c0'
Saving residual: shape=torch.Size([4, 512, 1]), dtype=torch.float32, grad_fn=None, ptr='0x2ea7f9c0'
Loading residual: shape=torch.Size([4, 512, 2560]), dtype=torch.float32, grad_fn=None, ptr='0x31bb3c80'
Loading residual: shape=torch.Size([2560]), dtype=torch.float32, grad_fn=None, ptr='0x2ec6b3c0'
Loading residual: shape=torch.Size([4, 512, 1]), dtype=torch.float32, grad_fn=None, ptr='0x2ea7f9c0'


In [20]:
# Now logs the number of bytes saved
total_size_bytes = 0
def pack_hook(t):
    if isinstance(t, torch.nn.Parameter): # Skip logging parameters to avoid double counting
        return t
    global total_size_bytes
    shape, dtype, grad_fn = t.shape, t.dtype, t.grad_fn
    total_size_bytes += t.numel() * t.element_size()
    print(f"Saving residual: {shape=}, {dtype=}, {grad_fn=}")
    return t

## 3.2 Activation Checkpointing

xl 一层 TransformerBlock，`torch.compile(fullgraph=True)` 融合到极限后还为反向存多少。hw1 的 block 要求显式传 `mask` 和 `token_positions`（官方实现内部生成），所以比 PDF 多 4 MiB 的 mask。

In [21]:
import torch
from cs336_basics.model import RotaryEmbedding, TransformerBlock, build_attention_mask
# num_layers for this model is 32
d_model, d_ff, num_heads, context_length = 2560, 10240, 16, 2048
device = torch.device("cuda")
block = TransformerBlock(
    d_model=d_model, d_ff=d_ff, num_heads=num_heads,
    rope=RotaryEmbedding(theta=10000.0, d_k=d_model // num_heads, max_seq_len=context_length, device=device),
    device=device,
)
block = torch.compile(block, fullgraph=True)
x = torch.randn((4, context_length, d_model), device=device, requires_grad=True)
mask = build_attention_mask(context_length, device=device)
token_positions = torch.arange(context_length, device=device)

def free(*names):
    """释放上一个 cell 留下的大张量。notebook 变量是全局的：`y` 活着，它 autograd 图里那 3.6 GiB
    saved tensors 就活着；重跑同一个 cell 时新旧两份并存，four_blocks 直接撞墙。"""
    g = globals()
    for n in names:
        g.pop(n, None)
    torch.cuda.empty_cache()
    print(f"GPU 驻留 {torch.cuda.memory_allocated() / 2**30:.2f} GiB")


**单层**：预期 ≈ 3655 MiB（PDF 3651），其中 attention 的 S、P 两个 `[b,h,s,s]` 占 2 GiB，FFN 三个 `[b,s,d_ff]` 占 960 MiB。

In [22]:
free("y")
total_size_bytes = 0
with torch.autograd.graph.saved_tensors_hooks(pack_hook, unpack_hook):
    y = block(x, mask=mask, token_positions=token_positions)
print(f"Total size of saved tensors in single TransformerBlock: {total_size_bytes / (1024**2):.2f} MiB")

GPU 驻留 0.49 GiB
Saving residual: shape=torch.Size([4, 2048, 2560]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([2048]), dtype=torch.int64, grad_fn=None
Saving residual: shape=torch.Size([2048, 80]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([2048, 80]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([2]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([1, 1, 2048, 2048]), dtype=torch.bool, grad_fn=None
Saving residual: shape=torch.Size([4, 2048, 1]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([64, 2048, 2048]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([4, 16, 2048, 1]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([4, 16, 2048, 1]), dtype=torch.int64, grad_fn=None
Saving residual: shape=torch.Size([4, 16, 2048, 1]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([4, 16, 2048, 2048]), dtype=torch.floa

**四层不 checkpoint**：预期 4 × 3655 ≈ 14.6 GiB——xl@2048 fp32 的 32 层就是 114 GiB，参数侧再省也没用。

In [24]:
free("y")
total_size_bytes = 0
def four_blocks(x):
    for _ in range(4):
        x = block(x, mask=mask, token_positions=token_positions)
    return x
with torch.autograd.graph.saved_tensors_hooks(pack_hook, unpack_hook):
    y = four_blocks(x)
print(f"Total size of saved tensors in four TransformerBlocks: {total_size_bytes / (1024**2):.2f} MiB")

GPU 驻留 0.49 GiB
Saving residual: shape=torch.Size([4, 2048, 2560]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([2048]), dtype=torch.int64, grad_fn=None
Saving residual: shape=torch.Size([2048, 80]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([2048, 80]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([2]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([1, 1, 2048, 2048]), dtype=torch.bool, grad_fn=None
Saving residual: shape=torch.Size([4, 2048, 1]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([64, 2048, 2048]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([4, 16, 2048, 1]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([4, 16, 2048, 1]), dtype=torch.int64, grad_fn=None
Saving residual: shape=torch.Size([4, 16, 2048, 1]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([4, 16, 2048, 2048]), dtype=torch.floa

**四层、每两层一个 checkpoint**：`checkpoint(fn, x, use_reentrant=False)` 前向只存 fn 的输入、不存内部 residual；反向到这一段时先重跑一遍前向把 residual 造出来，再正常反向，之后全部释放。
预期只剩两个入口的 `[4,2048,2560]` = 160 MiB。显存没有消失，只是拆成「长期：checkpoint 入口」和「短期：反向时重算一段的 residual」——峰值 = 入口总量 + 一段的 residual，段数是两者的权衡（§3.1(a) 的推导）。

In [25]:
from torch.utils.checkpoint import checkpoint
free("y")
total_size_bytes = 0
def two_blocks(x):
    x = block(x, mask=mask, token_positions=token_positions)
    x = block(x, mask=mask, token_positions=token_positions)
    return x
def four_blocks_checkpoint(x):
    x = checkpoint(two_blocks, x, use_reentrant=False)
    x = checkpoint(two_blocks, x, use_reentrant=False)
    return x
with torch.autograd.graph.saved_tensors_hooks(pack_hook, unpack_hook):
    y = four_blocks_checkpoint(x)
print(f"Total size of saved tensors in four TransformerBlocks with checkpointing: {total_size_bytes / (1024**2):.2f} MiB")

GPU 驻留 0.49 GiB
Saving residual: shape=torch.Size([0]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([4, 2048, 2560]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([0]), dtype=torch.float32, grad_fn=None
Saving residual: shape=torch.Size([4, 2048, 2560]), dtype=torch.float32, grad_fn=<torch.autograd.function.CompiledFunctionBackward object at 0x7325ceb26ad0>
Total size of saved tensors in four TransformerBlocks with checkpointing: 160.00 MiB


#### 3.2 (b) 平切段长扫描：large，fwd_bwd，batch 1，seq 1024

不嵌套 → 平切 k 段，峰值 ≈ k × entry + (L/k) × 一层 saved tensors；`checkpoint_every` = 每段几层（= L/k）。
题面 xl@2048 batch 4 在 5090 上任何段长都 OOM（参数+梯度 25.4 GiB）；large@2048 batch 4 也只有 every ≤ 4 能跑。要让全部段长含「不 checkpoint」都出数，降到 large / batch 1 / seq 1024（一层 saved tensors ≈ 0.26 GiB，不 checkpoint ≈ 17 GiB）。

⚠️ **重启 kernel，只跑 setup 和这两格**。`isolate=True` 每个配置独立进程。

In [3]:
run_config("checkpoint_large", f"{OUT}/checkpoint_large_b1_seq1024.md", isolate=True)

已写出：notes/assets/s3/checkpoint_large_b1_seq1024.md          
         notes/assets/s3/checkpoint_large_b1_seq1024.json（含逐步原始耗时）


,model,size,seq_len,batch,warmup,steps,mode,inference,autocast,avg_ms,std_ms,first_ms,rest_avg_ms,peak_mem_gib,status,checkpoint_every
0,basics,large,1024,1,2,5,fwd_bwd,False,False,235.87,0.53,235.82,235.88,15.02,OK,NaN
1,basics,large,1024,1,2,5,fwd_bwd,False,False,302.10,1.13,301.29,302.30,7.82,OK,1.0
2,basics,large,1024,1,2,5,fwd_bwd,False,False,305.58,0.48,305.52,305.60,8.03,OK,2.0
3,basics,large,1024,1,2,5,fwd_bwd,False,False,310.69,1.15,311.02,310.61,8.23,OK,3.0
4,basics,large,1024,1,2,5,fwd_bwd,False,False,310.54,0.74,310.38,310.58,8.44,OK,4.0
5,basics,large,1024,1,2,5,fwd_bwd,False,False,312.21,0.86,310.94,312.53,8.85,OK,6.0
6,basics,large,1024,1,2,5,fwd_bwd,False,False,309.34,1.04,308.95,309.43,9.47,OK,9.0
7,basics,large,1024,1,2,5,fwd_bwd,False,False,312.34,1.08,311.85,312.47,10.09,OK,12.0
8,basics,large,1024,1,2,5,fwd_bwd,False,False,313.10,1.38,312.90,313.15,11.32,OK,18.0
9,basics,large,1024,1,2,5,fwd_bwd,False,False,313.34,0.44,313.17,313.38,15.03,OK,36.0


In [ ]:
# 画图：x = 每段几层（对数轴），折线只连 checkpoint 的点；不 checkpoint 画成虚线基准
import json, matplotlib.pyplot as plt
rows = json.load(open(f"{OUT}/checkpoint_large_b1_seq1024.json"))
base = next(r for r in rows if r["checkpoint_every"] is None)
pts  = sorted((r for r in rows if r["checkpoint_every"]), key=lambda r: r["checkpoint_every"])
e   = [r["checkpoint_every"] for r in pts]
ms  = [r["avg_ms"] for r in pts]
gib = [r["peak_mem_gib"] for r in pts]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4), dpi=150)
a1.plot(e, ms, "-o", color="#2f6fdd", label="with checkpointing")
a1.axhline(base["avg_ms"], ls="--", color="#c62828", label=f"no checkpoint: {base['avg_ms']:.0f} ms")
a1.set_ylabel("fwd_bwd step (ms)"); a1.set_title("Step time vs layers per segment"); a1.set_ylim(0, 340)
a2.plot(e, gib, "-o", color="#2f6fdd", label="with checkpointing")
a2.axhline(base["peak_mem_gib"], ls="--", color="#c62828", label=f"no checkpoint: {base['peak_mem_gib']:.1f} GiB")
a2.set_ylabel("peak memory (GiB)"); a2.set_title("Peak memory vs layers per segment"); a2.set_ylim(0, 16)
for ax in (a1, a2):
    ax.set_xscale("log", base=2); ax.set_xticks(e); ax.set_xticklabels([str(v) for v in e]); ax.minorticks_off()
    ax.set_xlabel("layers per checkpoint segment"); ax.grid(color="#eee"); ax.spines[["top", "right"]].set_visible(False)
    ax.legend(loc="lower right", fontsize=8, frameon=False)
fig.suptitle("large (36 layers), batch 1, seq 1024, fp32 eager", fontsize=9)
fig.tight_layout(); fig.savefig(f"{OUT}/checkpoint_large_sweep.png", bbox_inches="tight"); plt.show()
